# NumPy Random Module

NumPy provides robust random number generation for simulations, Monte Carlo methods, and ML data shuffling.

**Topics:** `rand, randn, randint, uniform, normal, binomial, choice, shuffle, permutation`, the new Generator API, reproducibility.

## Legacy vs New Generator API

NumPy 1.17+ introduced a new random API via `Generator` (recommended). The legacy `np.random.*` functions still work but are frozen.

**Always prefer:** `rng = np.random.default_rng(seed)` over `np.random.seed(seed)`

In [ ]:
import numpy as np

# New recommended approach
rng = np.random.default_rng(42)  # seed for reproducibility
print("Generator type:", type(rng))

# Legacy approach (still works)
np.random.seed(42)
print("Legacy random:", np.random.rand(3))

## Common Distributions

In [ ]:
rng = np.random.default_rng(42)

# Uniform [0,1)
uni = rng.random((3, 4))
print("Uniform [0,1):\n", uni)

# randint (discrete uniform)
ints = rng.integers(0, 10, size=(2, 5))  # [0, 10)
print("\nIntegers [0,10):\n", ints)

# Uniform distribution (any range)
uniform = rng.uniform(5, 15, size=10)
print("\nUniform [5,15]:", uniform)

# Normal distribution
normal = rng.normal(loc=0, scale=1, size=1000)
print(f"\nNormal(0,1): mean={np.mean(normal):.3f}, std={np.std(normal):.3f}")

# Binomial
binom = rng.binomial(n=10, p=0.5, size=20)
print("Binomial(n=10, p=0.5):", binom)

## Choice, Shuffle, Permutation

In [ ]:
rng = np.random.default_rng(42)
arr = np.arange(10)

# Random choice with replacement
chosen = rng.choice(arr, size=5)
print("Choice (replace=True):", chosen)

# Choice without replacement
chosen_no = rng.choice(arr, size=5, replace=False)
print("Choice (replace=False):", chosen_no)

# Choice with probabilities
weighted = rng.choice(['A', 'B', 'C'], size=10, p=[0.5, 0.3, 0.2])
print("\nWeighted choice:", weighted)

# Shuffle (in-place)
deck = np.arange(52)
rng.shuffle(deck)
print("\nShuffled deck (first 10):", deck[:10])

# Permutation (returns new array)
arr2 = np.arange(10)
perm = rng.permutation(arr2)
print("Permutation:", perm)
print("Original unchanged:", arr2)

## Reproducibility

Same seed → same sequence. Essential for reproducible research.

In [ ]:
# Same seed gives same results
rng1 = np.random.default_rng(123)
rng2 = np.random.default_rng(123)

a = rng1.random(5)
b = rng2.random(5)
print("rng1:", a)
print("rng2:", b)
print("Same?", np.allclose(a, b))

# Different seed → different results
rng3 = np.random.default_rng(456)
c = rng3.random(5)
print("\nrng3 (seed=456):", c)
print("Different?", not np.allclose(a, c))

## Problem 1: Monte Carlo Simulation for Pi Estimation

Throw random points into a unit square, count how many fall inside the unit quarter-circle.

In [ ]:
def estimate_pi(num_points):
    rng = np.random.default_rng(42)
    x = rng.random(num_points)
    y = rng.random(num_points)
    # Points inside quarter circle: x^2 + y^2 <= 1
    inside = np.sum(x**2 + y**2 <= 1)
    return 4 * inside / num_points

for n in [100, 1_000, 10_000, 100_000, 1_000_000]:
    pi_est = estimate_pi(n)
    error = abs(pi_est - np.pi)
    print(f"n={n:>8d}: pi ≈ {pi_est:.6f}  (error={error:.6f})")

# Law of large numbers: more points → better estimate

## Problem 2: Random Walk Simulation

Simulate a 1D random walk of 10,000 steps and compute statistics.

In [ ]:
def random_walk(n_steps, seed=42):
    rng = np.random.default_rng(seed)
    steps = rng.choice([-1, 1], size=n_steps)
    position = np.concatenate([[0], np.cumsum(steps)])
    return position

walk = random_walk(10000)

print(f"Steps: {len(walk)}")
print(f"Final position: {walk[-1]}")
print(f"Max position: {np.max(walk)}")
print(f"Min position: {np.min(walk)}")
print(f"Mean position: {np.mean(walk):.2f}")
print(f"Std position: {np.std(walk):.2f}")
print(f"Std expected (sqrt(n)): {np.sqrt(10000):.2f}")

# 2D random walk
def random_walk_2d(n_steps, seed=42):
    rng = np.random.default_rng(seed)
    angles = rng.uniform(0, 2*np.pi, n_steps)
    dx = np.cos(angles)
    dy = np.sin(angles)
    x = np.concatenate([[0], np.cumsum(dx)])
    y = np.concatenate([[0], np.cumsum(dy)])
    return x, y

x, y = random_walk_2d(5000)
dist = np.sqrt(x[-1]**2 + y[-1]**2)
print(f"\n2D Walk: final distance from origin: {dist:.2f}")

## Problem 3: Monte Carlo Integration

Estimate the integral of `f(x) = sin(x)` over `[0, π]` using Monte Carlo.

In [ ]:
def mc_integrate(f, a, b, n_points):
    rng = np.random.default_rng(42)
    x = rng.uniform(a, b, n_points)
    fx = f(x)
    return (b - a) * np.mean(fx)

# Integrate sin(x) over [0, pi] (true value = 2)
for n in [100, 1_000, 10_000, 100_000]:
    est = mc_integrate(np.sin, 0, np.pi, n)
    print(f"n={n:>7d}: integral ≈ {est:.6f}  (error={abs(est-2):.6f})")

# Multiple dimensions — MC integration shines
# Volume of a hypersphere in 4D
def hypersphere_volume_4d(n_points):
    rng = np.random.default_rng(42)
    points = rng.uniform(-1, 1, (n_points, 4))
    inside = np.sum(np.sum(points**2, axis=1) <= 1)
    return (16) * inside / n_points  # hypercube volume = 2^4 = 16

vol = hypersphere_volume_4d(500_000)
print(f"\nHypersphere volume (4D): {vol:.4f}  (true: {np.pi**2 / 2:.4f})")

## Summary

- Use `np.random.default_rng(seed)` (Generator API) — not the legacy `np.random.seed()`
- Distributions: `random, integers, uniform, normal, binomial`
- Sampling: `choice` (with/without replacement, weighted), `shuffle` (in-place), `permutation`
- Seed for reproducibility → same seed = same sequence
- Monte Carlo: random sampling for integration, simulation, optimization